# Bitcoin Mining Pool Classifier - Data from BigQuery

## Imports

In [1]:
from google.cloud import bigquery
client = bigquery.Client()
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import itertools

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import  confusion_matrix, recall_score,precision_score, precision_recall_curve, f1_score, fbeta_score
from sklearn.utils.fixes import signature


Using Kaggle's public dataset BigQuery integration.


## Load data from BigQuery

Note: We query a subset of the data here due to Kaggle resource constraints

In [11]:
from google.cloud import bigquery
client = bigquery.Client()

query = """
SELECT
    *
FROM
  `bigquery-public-data.crypto_dogecoin.blocks`
ORDER BY
  number
LIMIT 10
"""

df = client.query(query).to_dataframe()
df


Using Kaggle's public dataset BigQuery integration.


,hash,size,stripped_size,weight,number,version,merkle_root,timestamp,timestamp_month,nonce,bits,coinbase_param,transaction_count
0,1a91e3dace36e2be3bf030a65679fe821aa1d6ef92e7c9...,224,224,896,0,1,5b2a3f53f605d62c53e62932dac6925e3d74afa5a4b459...,2013-12-06 10:25:40+00:00,2013-12-01,18667,1e0ffff0,04ffff001d0104455468652054696d65732030332f4a61...,1
1,82bc68038f6034c0596b6e313729793a887fded6e92a31...,190,190,760,1,1,5f7e779f7600f54e528686e91d5891f3ae226ee907f461...,2013-12-08 03:55:27+00:00,2013-12-01,54831000,1e0ffff0,04afeda3520102062f503253482f,1
2,ea5380659e02a68c073369e502125c634b2fb0aaf351b9...,190,190,760,2,1,3b14b76d22a3f2859d73316002bc1b9bfc7f37e2c3393b...,2013-12-08 03:55:33+00:00,2013-12-01,cae81700,1e0ffff0,04b5eda3520101062f503253482f,1
3,76f80a8a81e6f6669d340651723b874f97395c4dbda200...,190,190,760,3,1,1e10c28574e3b9d7032329b624ce4ac8064d0e91324aa1...,2013-12-08 03:55:40+00:00,2013-12-01,e1a00700,1e0ffff0,04bceda3520101062f503253482f,1
4,df363f95151d8c38b1cf0ee8375d571c9a869d9e37489b...,190,190,760,4,1,9f69a09b940fc7645b0a261e81a1f777e3e6514989eaf1...,2013-12-08 03:55:43+00:00,2013-12-01,9021200,1e0ffff0,04bfeda3520102062f503253482f,1
5,f21dc70cb44c180261e31a222202678602d605e7697332...,190,190,760,5,1,bd19df265c7db76eb4a03a672515896014da663bc39931...,2013-12-08 03:55:49+00:00,2013-12-01,a5400900,1e0ffff0,04c5eda3520101062f503253482f,1
6,f34986a114a2f58f48ce5593e5e6006666243fb003a2ff...,190,190,760,6,1,de82cb962ff643ddaa0069356d02fd08f5e4c30105445f...,2013-12-08 03:55:52+00:00,2013-12-01,55570c00,1e0ffff0,04c8eda3520102062f503253482f,1
7,3ca7e813da5c72b0817c4d4789cd4896f49baf5e40f67a...,190,190,760,7,1,caa91617f88611e898ecfb23fb95890c875491dee1d53a...,2013-12-08 03:55:57+00:00,2013-12-01,91581d00,1e0ffff0,04cdeda3520102062f503253482f,1
8,ed6b216e69b57915eda3a43036016c5667b35d61606f1c...,190,190,760,8,1,69f88eaff52611504a40c28e5353d2b5ad7e734871b19d...,2013-12-08 03:56:07+00:00,2013-12-01,381e0300,1e0ffff0,04d7eda3520102062f503253482f,1
9,0ddd48852cb794c7534841a2cd3507e40255b1707fac14...,190,190,760,9,1,d127140404893d7fbb560fe0f356661df1fae0d2332ac3...,2013-12-08 03:56:09+00:00,2013-12-01,79740300,1e0ffff0,04d9eda3520101062f503253482f,1


In [3]:
from google.cloud import bigquery
client = bigquery.Client()

query = """
SELECT
    *
FROM
  `bigquery-public-data.crypto_bitcoin.transactions`
ORDER BY
  block_timestamp
LIMIT 10
"""

df = client.query(query).to_dataframe()
df


Using Kaggle's public dataset BigQuery integration.


,hash,size,virtual_size,version,lock_time,block_hash,block_number,block_timestamp,block_timestamp_month,input_count,output_count,input_value,output_value,is_coinbase,fee,inputs,outputs
0,4a5e1e4baab89f3a32518a88c31bc87f618f76673e2cc7...,204,204,1,0,000000000019d6689c085ae165831e934ff763ae46a2a6...,0,2009-01-03 18:15:05+00:00,2009-01-01,0,1,None,5000000000,True,0,[],"[{'index': 0, 'script_asm': '04678afdb0fe55482..."
1,0e3e2357e806b6cdb1f70b54c3a3a17b6714ee1f0e68be...,134,134,1,0,00000000839a8e6886ab5951d76f411475428afc90947e...,1,2009-01-09 02:54:25+00:00,2009-01-01,0,1,None,5000000000,True,0,[],"[{'index': 0, 'script_asm': '0496b538e853519c7..."
2,9b0fc92260312ce44e74ef369f5c66bbb85848f2eddd5a...,134,134,1,0,000000006a625f06636b8bb6ac7b960a8d03705d1ace08...,2,2009-01-09 02:55:44+00:00,2009-01-01,0,1,None,5000000000,True,0,[],"[{'index': 0, 'script_asm': '047211a824f55b505..."
3,999e1c837c76a1b7fbb7e57baf87b309960f5ffefbf2a9...,134,134,1,0,0000000082b5015589a3fdf2d4baff403e6f0be035a5d9...,3,2009-01-09 03:02:53+00:00,2009-01-01,0,1,None,5000000000,True,0,[],"[{'index': 0, 'script_asm': '0494b9d3e76c5b162..."
4,df2b060fa2e5e9c8ed5eaf6a45c13753ec8c63282b2688...,134,134,1,0,000000004ebadb55ee9096c9a2f8880e09da59c0d68b1c...,4,2009-01-09 03:16:28+00:00,2009-01-01,0,1,None,5000000000,True,0,[],"[{'index': 0, 'script_asm': '04184f32b212815c6..."
5,63522845d294ee9b0188ae5cac91bf389a0c3723f084ca...,134,134,1,0,000000009b7262315dbf071787ad3656097b892abffd1f...,5,2009-01-09 03:23:48+00:00,2009-01-01,0,1,None,5000000000,True,0,[],"[{'index': 0, 'script_asm': '0456579536d150fbc..."
6,20251a76e64e920e58291a30d4b212939aae976baca40e...,134,134,1,0,000000003031a0e73735690c5a1ff2a4be82553b2a12b7...,6,2009-01-09 03:29:49+00:00,2009-01-01,0,1,None,5000000000,True,0,[],"[{'index': 0, 'script_asm': '0408ce279174b34c0..."
7,8aa673bc752f2851fd645d6a0a92917e967083007d9c16...,134,134,1,0,0000000071966c2b1d065fd446b1e485b2c9d9594acd20...,7,2009-01-09 03:39:29+00:00,2009-01-01,0,1,None,5000000000,True,0,[],"[{'index': 0, 'script_asm': '04a59e64c774923d0..."
8,a6f7f1c0dad0f2eb6b13c4f33de664b1b0e9f22efad599...,134,134,1,0,00000000408c48f847aa786c2268fc3e6ec2af68e8468a...,8,2009-01-09 03:45:43+00:00,2009-01-01,0,1,None,5000000000,True,0,[],"[{'index': 0, 'script_asm': '04cc8d85f5e7933cb..."
9,0437cd7f8525ceed2324359c2d0ba26006d92d856a9c20...,134,134,1,0,000000008d9dc510f23c2657fc4f67bea30078cc05a90e...,9,2009-01-09 03:54:39+00:00,2009-01-01,0,1,None,5000000000,True,0,[],"[{'index': 0, 'script_asm': '0411db93e1dcdb8a0..."


In [2]:
table = client.get_table("bigquery-public-data.crypto_bitcoin.transactions")
print([schema.name for schema in table.schema])

['hash', 'size', 'virtual_size', 'version', 'lock_time', 'block_hash', 'block_number', 'block_timestamp', 'block_timestamp_month', 'input_count', 'output_count', 'input_value', 'output_value', 'is_coinbase', 'fee', 'inputs', 'outputs']


In [ ]:
miner_vectors_limit = 2000
non_miner_vectors_limit = 20000

In [ ]:
sql='''
WITH 
output_ages AS (
  SELECT
    ARRAY_TO_STRING(outputs.addresses,',') AS output_ages_address,
    MIN(block_timestamp_month) AS output_month_min,
    MAX(block_timestamp_month) AS output_month_max
  FROM `bigquery-public-data.crypto_bitcoin.transactions` AS transactions JOIN UNNEST(outputs) AS outputs
  GROUP BY output_ages_address
)
,input_ages AS (
  SELECT
    ARRAY_TO_STRING(inputs.addresses,',') AS input_ages_address,
    MIN(block_timestamp_month) AS input_month_min,
    MAX(block_timestamp_month) AS input_month_max
  FROM `bigquery-public-data.crypto_bitcoin.transactions` AS transactions JOIN UNNEST(inputs) AS inputs
  GROUP BY input_ages_address
)
,output_monthly_stats AS (
  SELECT
    ARRAY_TO_STRING(outputs.addresses,',') AS output_monthly_stats_address, 
    COUNT(DISTINCT block_timestamp_month) AS output_active_months,
    COUNT(outputs) AS total_tx_output_count,
    SUM(value) AS total_tx_output_value,
    AVG(value) AS mean_tx_output_value,
    STDDEV(value) AS stddev_tx_output_value,
    COUNT(DISTINCT(`hash`)) AS total_output_tx,
    SUM(value)/COUNT(block_timestamp_month) AS mean_monthly_output_value,
    COUNT(outputs.addresses)/COUNT(block_timestamp_month) AS mean_monthly_output_count
  FROM `bigquery-public-data.crypto_bitcoin.transactions` AS transactions JOIN UNNEST(outputs) AS outputs
  GROUP BY output_monthly_stats_address
)
,input_monthly_stats AS (
  SELECT
    ARRAY_TO_STRING(inputs.addresses,',') AS input_monthly_stats_address, 
    COUNT(DISTINCT block_timestamp_month) AS input_active_months,
    COUNT(inputs) AS total_tx_input_count,
    SUM(value) AS total_tx_input_value,
    AVG(value) AS mean_tx_input_value,
    STDDEV(value) AS stddev_tx_input_value,
    COUNT(DISTINCT(`hash`)) AS total_input_tx,
    SUM(value)/COUNT(block_timestamp_month) AS mean_monthly_input_value,
    COUNT(inputs.addresses)/COUNT(block_timestamp_month) AS mean_monthly_input_count
  FROM `bigquery-public-data.crypto_bitcoin.transactions` AS transactions JOIN UNNEST(inputs) AS inputs
  GROUP BY input_monthly_stats_address
)
,output_idle_times AS (
  SELECT
    address AS idle_time_address,
    AVG(idle_time) AS mean_output_idle_time,
    STDDEV(idle_time) AS stddev_output_idle_time
  FROM
  (
    SELECT 
      event.address,
      IF(prev_block_time IS NULL, NULL, UNIX_SECONDS(block_time) - UNIX_SECONDS(prev_block_time)) AS idle_time
    FROM (
      SELECT
        ARRAY_TO_STRING(outputs.addresses,',') AS address, 
        block_timestamp AS block_time,
        LAG(block_timestamp) OVER (PARTITION BY ARRAY_TO_STRING(outputs.addresses,',') ORDER BY block_timestamp) AS prev_block_time
      FROM `bigquery-public-data.crypto_bitcoin.transactions` AS transactions JOIN UNNEST(outputs) AS outputs
    ) AS event
    WHERE block_time != prev_block_time
  )
  GROUP BY address
)
,input_idle_times AS (
  SELECT
    address AS idle_time_address,
    AVG(idle_time) AS mean_input_idle_time,
    STDDEV(idle_time) AS stddev_input_idle_time
  FROM
  (
    SELECT 
      event.address,
      IF(prev_block_time IS NULL, NULL, UNIX_SECONDS(block_time) - UNIX_SECONDS(prev_block_time)) AS idle_time
    FROM (
      SELECT
        ARRAY_TO_STRING(inputs.addresses,',') AS address, 
        block_timestamp AS block_time,
        LAG(block_timestamp) OVER (PARTITION BY ARRAY_TO_STRING(inputs.addresses,',') ORDER BY block_timestamp) AS prev_block_time
      FROM `bigquery-public-data.crypto_bitcoin.transactions` AS transactions JOIN UNNEST(inputs) AS inputs
    ) AS event
    WHERE block_time != prev_block_time
  )
  GROUP BY address
)
--,miners AS (
--)

(SELECT
  TRUE AS is_miner,
  output_ages_address AS address,
  UNIX_SECONDS(CAST(output_ages.output_month_min AS TIMESTAMP)) AS output_month_min,
  UNIX_SECONDS(CAST(output_ages.output_month_max AS TIMESTAMP)) AS output_month_max,
  UNIX_SECONDS(CAST(input_ages.input_month_min AS TIMESTAMP)) AS input_month_min,
  UNIX_SECONDS(CAST(input_ages.input_month_max AS TIMESTAMP)) AS input_month_max,
  UNIX_SECONDS(CAST(output_ages.output_month_max AS TIMESTAMP)) - UNIX_SECONDS(CAST(output_ages.output_month_min AS TIMESTAMP)) AS output_active_time,
  UNIX_SECONDS(CAST(input_ages.input_month_max AS TIMESTAMP)) - UNIX_SECONDS(CAST(input_ages.input_month_min AS TIMESTAMP)) AS input_active_time,
  UNIX_SECONDS(CAST(output_ages.output_month_max AS TIMESTAMP)) - UNIX_SECONDS(CAST(input_ages.input_month_max AS TIMESTAMP)) AS io_max_lag,
  UNIX_SECONDS(CAST(output_ages.output_month_min AS TIMESTAMP)) - UNIX_SECONDS(CAST(input_ages.input_month_min AS TIMESTAMP)) AS io_min_lag,
  output_monthly_stats.output_active_months,
  output_monthly_stats.total_tx_output_count,
  output_monthly_stats.total_tx_output_value,
  output_monthly_stats.mean_tx_output_value,
  output_monthly_stats.stddev_tx_output_value,
  output_monthly_stats.total_output_tx,
  output_monthly_stats.mean_monthly_output_value,
  output_monthly_stats.mean_monthly_output_count,
  input_monthly_stats.input_active_months,
  input_monthly_stats.total_tx_input_count,
  input_monthly_stats.total_tx_input_value,
  input_monthly_stats.mean_tx_input_value,
  input_monthly_stats.stddev_tx_input_value,
  input_monthly_stats.total_input_tx,
  input_monthly_stats.mean_monthly_input_value,
  input_monthly_stats.mean_monthly_input_count,
  output_idle_times.mean_output_idle_time,
  output_idle_times.stddev_output_idle_time,
  input_idle_times.mean_input_idle_time,
  input_idle_times.stddev_input_idle_time
FROM
  output_ages, output_monthly_stats, output_idle_times,
  input_ages,  input_monthly_stats, input_idle_times
WHERE TRUE
  AND output_ages.output_ages_address = output_monthly_stats.output_monthly_stats_address
  AND output_ages.output_ages_address = output_idle_times.idle_time_address
  AND output_ages.output_ages_address = input_monthly_stats.input_monthly_stats_address
  AND output_ages.output_ages_address = input_ages.input_ages_address
  AND output_ages.output_ages_address = input_idle_times.idle_time_address
  AND output_ages.output_ages_address IN
(
  SELECT 
    ARRAY_TO_STRING(outputs.addresses,',') AS miner
  FROM 
  `bigquery-public-data.crypto_bitcoin.blocks` AS blocks,
  `bigquery-public-data.crypto_bitcoin.transactions` AS transactions JOIN UNNEST(outputs) AS outputs
  WHERE blocks.hash = transactions.block_hash 
    AND is_coinbase IS TRUE
    AND ( FALSE
      --
      -- miner signatures from https://en.bitcoin.it/wiki/Comparison_of_mining_pools
      --
      OR coinbase_param LIKE '%4d696e656420627920416e74506f6f6c%' --AntPool
      OR coinbase_param LIKE '%2f42434d6f6e737465722f%' --BCMonster
      --BitcoinAffiliateNetwork
      OR coinbase_param LIKE '%4269744d696e746572%' --BitMinter
      --BTC.com
      --BTCC Pool
      --BTCDig
      OR coinbase_param LIKE '%2f7374726174756d2f%' --Btcmp
      --btcZPool.com
      --BW Mining
      OR coinbase_param LIKE '%456c6967697573%' --Eligius
      --F2Pool
      --GHash.IO
      --Give Me COINS
      --Golden Nonce Pool
      OR coinbase_param LIKE '%2f627261766f2d6d696e696e672f%' --Bravo Mining
      OR coinbase_param LIKE '%4b616e6f%' --KanoPool
      --kmdPool.org
      OR coinbase_param LIKE '%2f6d6d706f6f6c%' --Merge Mining Pool
      --MergeMining
      --Multipool
      --P2Pool
      OR coinbase_param LIKE '%2f736c7573682f%' --Slush Pool
      --ZenPool.org
    )
  GROUP BY miner
  HAVING COUNT(1) >= 20 
)
LIMIT {})
UNION ALL
(SELECT
  FALSE AS is_miner,
  output_ages_address AS address,
  UNIX_SECONDS(CAST(output_ages.output_month_min AS TIMESTAMP)) AS output_month_min,
  UNIX_SECONDS(CAST(output_ages.output_month_max AS TIMESTAMP)) AS output_month_max,
  UNIX_SECONDS(CAST(input_ages.input_month_min AS TIMESTAMP)) AS input_month_min,
  UNIX_SECONDS(CAST(input_ages.input_month_max AS TIMESTAMP)) AS input_month_max,
  UNIX_SECONDS(CAST(output_ages.output_month_max AS TIMESTAMP)) - UNIX_SECONDS(CAST(output_ages.output_month_min AS TIMESTAMP)) AS output_active_time,
  UNIX_SECONDS(CAST(input_ages.input_month_max AS TIMESTAMP)) - UNIX_SECONDS(CAST(input_ages.input_month_min AS TIMESTAMP)) AS input_active_time,
  UNIX_SECONDS(CAST(output_ages.output_month_max AS TIMESTAMP)) - UNIX_SECONDS(CAST(input_ages.input_month_max AS TIMESTAMP)) AS io_max_lag,
  UNIX_SECONDS(CAST(output_ages.output_month_min AS TIMESTAMP)) - UNIX_SECONDS(CAST(input_ages.input_month_min AS TIMESTAMP)) AS io_min_lag,
  output_monthly_stats.output_active_months,
  output_monthly_stats.total_tx_output_count,
  output_monthly_stats.total_tx_output_value,
  output_monthly_stats.mean_tx_output_value,
  output_monthly_stats.stddev_tx_output_value,
  output_monthly_stats.total_output_tx,
  output_monthly_stats.mean_monthly_output_value,
  output_monthly_stats.mean_monthly_output_count,
  input_monthly_stats.input_active_months,
  input_monthly_stats.total_tx_input_count,
  input_monthly_stats.total_tx_input_value,
  input_monthly_stats.mean_tx_input_value,
  input_monthly_stats.stddev_tx_input_value,
  input_monthly_stats.total_input_tx,
  input_monthly_stats.mean_monthly_input_value,
  input_monthly_stats.mean_monthly_input_count,
  output_idle_times.mean_output_idle_time,
  output_idle_times.stddev_output_idle_time,
  input_idle_times.mean_input_idle_time,
  input_idle_times.stddev_input_idle_time
FROM
  output_ages, output_monthly_stats, output_idle_times,
  input_ages,  input_monthly_stats, input_idle_times
WHERE TRUE
  AND output_ages.output_ages_address = output_monthly_stats.output_monthly_stats_address
  AND output_ages.output_ages_address = output_idle_times.idle_time_address
  AND output_ages.output_ages_address = input_monthly_stats.input_monthly_stats_address
  AND output_ages.output_ages_address = input_ages.input_ages_address
  AND output_ages.output_ages_address = input_idle_times.idle_time_address
  AND output_ages.output_ages_address NOT IN
(
  SELECT 
    ARRAY_TO_STRING(outputs.addresses,',') AS miner
  FROM 
  `bigquery-public-data.crypto_bitcoin.blocks` AS blocks,
  `bigquery-public-data.crypto_bitcoin.transactions` AS transactions JOIN UNNEST(outputs) AS outputs
  WHERE blocks.hash = transactions.block_hash 
    AND is_coinbase IS TRUE
    AND ( FALSE
      --
      -- miner signatures from https://en.bitcoin.it/wiki/Comparison_of_mining_pools
      --
      OR coinbase_param LIKE '%4d696e656420627920416e74506f6f6c%' --AntPool
      OR coinbase_param LIKE '%2f42434d6f6e737465722f%' --BCMonster
      --BitcoinAffiliateNetwork
      OR coinbase_param LIKE '%4269744d696e746572%' --BitMinter
      --BTC.com
      --BTCC Pool
      --BTCDig
      OR coinbase_param LIKE '%2f7374726174756d2f%' --Btcmp
      --btcZPool.com
      --BW Mining
      OR coinbase_param LIKE '%456c6967697573%' --Eligius
      --F2Pool
      --GHash.IO
      --Give Me COINS
      --Golden Nonce Pool
      OR coinbase_param LIKE '%2f627261766f2d6d696e696e672f%' --Bravo Mining
      OR coinbase_param LIKE '%4b616e6f%' --KanoPool
      --kmdPool.org
      OR coinbase_param LIKE '%2f6d6d706f6f6c%' --Merge Mining Pool
      --MergeMining
      --Multipool
      --P2Pool
      OR coinbase_param LIKE '%2f736c7573682f%' --Slush Pool
      --ZenPool.org
    )
  GROUP BY miner
  HAVING COUNT(1) >= 20 
)
LIMIT {})
'''.format(miner_vectors_limit, non_miner_vectors_limit)

In [ ]:
df = client.query(sql).to_dataframe()

In [ ]:
df.info()

In [ ]:
#drop columns with null values
df.drop(labels=['stddev_output_idle_time','stddev_input_idle_time'], axis=1, inplace=True)

## Split Data into Training Set and Test Set

In [ ]:
#get rid of non-numeric features
features = df.drop(labels=['is_miner','address'], axis=1)
target = df['is_miner'].values
indices = range(len(features))

#Train test split
X_train, X_test, y_train, y_test, indices_train, indices_test = train_test_split(features, target, indices,  test_size=0.2)

## Train a Model

In [ ]:
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced')
rf.fit(X_train, y_train)

## Make Predictions

In [ ]:
y_pred = rf.predict(X_test) #
probs = rf.predict_proba(X_test)[:,1] #positive class probabilities

## How good is our model?

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, probs)

In [ ]:
# Precision / recall curve code adapted from https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_recall_curve.html

fig, ax = plt.subplots(figsize=(8,6))
step_kwargs = ({'step': 'post'}
               if 'step' in signature(plt.fill_between).parameters
               else {})
plt.step(recall, precision, color='b', alpha=0.2,
         where='post')
plt.fill_between(recall, precision, alpha=0.2, color='b', **step_kwargs)

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.ylim([0.0, 1.0])
plt.xlim([0.0, 1.0])
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
plt.title('Mining Pool Detector - Precision/Recall Curve', fontsize=14)

In [ ]:
#confusion matrix code adapted from https://scikit-learn.org/stable/auto_examples/model_selection/plot_confusion_matrix.html

def plot_confusion_matrix(cm, classes,
                          normalize=False,
                          title='Confusion matrix',
                          cmap=plt.cm.Blues):
    """
    This function prints and plots the confusion matrix.
    Normalization can be applied by setting `normalize=True`.
    """
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')

    print(cm)
    dummy=np.array([[0,0],[0,0]])
    plt.figure(figsize=(8,6))
    plt.imshow(dummy, interpolation='nearest', cmap=cmap)
    plt.title(title)
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="black")

    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()


# Compute confusion matrix
cnf_matrix = confusion_matrix(y_test, y_pred)
class_names = ['not mining pool', 'mining pool']
np.set_printoptions(precision=2)

# Plot confusion matrix
plt.figure()
plot_confusion_matrix(cnf_matrix, classes=class_names, normalize=False,
                      title='Mining Pool Detector - Confusion Matrix')

plt.show()

## What features provide the most signal?

In [ ]:
x_pos = np.arange(len(features.columns))
btc_importances = rf.feature_importances_

inds = np.argsort(btc_importances)[::-1]
btc_importances = btc_importances[inds]
cols = features.columns[inds]
bar_width = .8

#how many features to plot?
n_features=12
x_pos = x_pos[:n_features][::-1]
btc_importances = btc_importances[:n_features]

#plot
plt.figure(figsize=(12,6))
plt.barh(x_pos, btc_importances, bar_width, label='BTC model')
plt.yticks(x_pos, cols, rotation=0, fontsize=14)
plt.xlabel('feature importance', fontsize=14)
plt.title('Mining Pool Detector', fontsize=20)
plt.tight_layout()

## Are False Positives associated with dark mining pools?

In [ ]:
#data points where model predicts true, but are labelled as false
false_positives = (y_test==False) & (y_pred==True)

In [ ]:
#subset to test set data only
df_test = df.iloc[indices_test, :]

print('False Positive addresses')

#subset test set to false positives only
df_test.iloc[false_positives].head(15)